In [ ]:
################## K - means ( Manon )
import pandas as pd
import folium
import joblib
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)

# =====================================================
# 1. CHARGEMENT ET PRÉPARATION DES DONNÉES
# =====================================================

df = pd.read_csv("export_IA.csv")

colonnes = [
    "consolidated_latitude",
    "consolidated_longitude"
]

df = df[colonnes].dropna()

df["consolidated_latitude"] = pd.to_numeric(
    df["consolidated_latitude"],
    errors="coerce"
)

df["consolidated_longitude"] = pd.to_numeric(
    df["consolidated_longitude"],
    errors="coerce"
)

df = df.dropna()

# Filtrage France métropolitaine
df = df[
    (df["consolidated_longitude"] >= -5.5) &
    (df["consolidated_longitude"] <= 9.5) &
    (df["consolidated_latitude"] >= 41) &
    (df["consolidated_latitude"] <= 51.5)
]

print("Nombre de bornes utilisées :", len(df))

X = df[
    ["consolidated_latitude", "consolidated_longitude"]
].values

# =====================================================
# CLUSTERING K-MEANS SELON LA POSITION 
# =====================================================

# 1. Définition des features et normalisation
features = ["consolidated_latitude", "consolidated_longitude"]
X = df[features]
# remet la latitude et longitude sur une échelle comparable
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. Entraînement du modèle
# On demande 5 groupes, il essaie 10 fois et garde le meilleur résultat
k = 5
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled) # C'est cette ligne qui manquait !
# Reconvertir les centres en coordonnées géographiques réelles
centers_geo = scaler.inverse_transform(kmeans.cluster_centers_)
# centers_geo[:, 0] = latitudes, centers_geo[:, 1] = longitudes
# 3. Affichage graphique
plt.figure(figsize=(10, 8))
for cluster_id in range(k):
    subset = df[df["cluster"] == cluster_id]
    plt.scatter(
        subset["consolidated_longitude"],
        subset["consolidated_latitude"],
        label=f"Cluster {cluster_id}",
        s=5,
        alpha=0.6
    )
# Croix des centres dans le bon repère
plt.scatter(
    centers_geo[:, 1],  # longitudes
    centers_geo[:, 0],  # latitudes
    marker='X', s=200, c='red', edgecolor='black',
    zorder=5, label='Centres des clusters'
)
plt.title("Clustering K-Means des bornes selon leur position")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend(title="Groupes de stations", loc="best")
#plt.show()
plt.savefig("carte_clusteringKMeans.png", dpi=150, bbox_inches="tight")

# 4. Analyse
print(df["cluster"].value_counts())
print(df.groupby("cluster")[features].mean())



# PARTIE ######################## AYA #####################

# =====================================================
# 1. CHARGEMENT DES DONNÉES
# =====================================================

# Chargement du fichier CSV nettoyé issu de la partie Big Data
df = pd.read_csv("export_IA.csv")

# Affichage du nombre de lignes avant traitement
print("Nombre de lignes initial :", len(df))

# =====================================================
# 2. SÉLECTION DES VARIABLES
# =====================================================

# Pour le clustering géographique, seules les coordonnées sont nécessaires
# Le modèle va regrouper les bornes selon leur proximité spatiale
df = df[
    [
        "consolidated_latitude",
        "consolidated_longitude"
    ]
]

# =====================================================
# 3. NETTOYAGE
# =====================================================

# Suppression des lignes avec latitude ou longitude manquante
df = df.dropna()

# Conversion de la latitude en format numérique
df["consolidated_latitude"] = pd.to_numeric(
    df["consolidated_latitude"],
    errors="coerce"
)

# Conversion de la longitude en format numérique
df["consolidated_longitude"] = pd.to_numeric(
    df["consolidated_longitude"],
    errors="coerce"
)

# Suppression des lignes qui auraient encore des valeurs invalides après conversion
df = df.dropna()

# =====================================================
# 4. FILTRAGE FRANCE MÉTROPOLITAINE
# =====================================================

# Filtrage des coordonnées pour garder uniquement les bornes situées en France métropolitaine
# Cela permet de supprimer les coordonnées incohérentes ou hors zone d’étude
df = df[
    (df["consolidated_longitude"] >= -5.5) &
    (df["consolidated_longitude"] <= 9.5) &
    (df["consolidated_latitude"] >= 41) &
    (df["consolidated_latitude"] <= 51.5)
]

# Affichage du nombre de lignes après nettoyage
print("Nombre de lignes après nettoyage :", len(df))

# =====================================================
# 5. MATRICE POUR K-MEANS
# =====================================================

# Création de la matrice X utilisée par K-Means
# Elle contient uniquement latitude et longitude
X = df[
    [
        "consolidated_latitude",
        "consolidated_longitude"
    ]
]

# Affichage des premières lignes pour vérifier les données utilisées
print("\nAperçu des données utilisées :")
print(X.head())

# Affichage de la taille de la matrice : nombre de bornes, nombre de variables
print("\nDimensions de la matrice :")
print(X.shape)

# =====================================================
# 6. MÉTHODE DU COUDE
# =====================================================

from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# Liste qui va stocker l'inertie pour chaque valeur de K
inerties = []

# On teste plusieurs nombres de clusters entre 2 et 15
K_range = range(2, 16)

for k in K_range:

    # Création d'un modèle K-Means avec k clusters
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    # Entraînement du modèle sur les coordonnées
    kmeans.fit(X)

    # Stockage de l'inertie
    # L'inertie mesure la distance des points à leur centre de cluster
    inerties.append(kmeans.inertia_)

# =====================================================
# 7. GRAPHIQUE DU COUDE
# =====================================================

# Création du graphique permettant de visualiser l'évolution de l'inertie
plt.figure(figsize=(8, 5))

plt.plot(
    K_range,
    inerties,
    marker="o"
)

plt.title("Méthode du coude")
plt.xlabel("Nombre de clusters (K)")
plt.ylabel("Inertie")

plt.grid(True)

# Affichage du graphique
plt.show()

# =====================================================
# 8. MÉTRIQUES D'ÉVALUATION DES CLUSTERS
# =====================================================

from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)

print("\nÉvaluation des différents K :\n")

for k in range(2, 16):

    # Création d'un K-Means avec k clusters
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    # Entraînement du modèle et attribution d'un cluster à chaque borne
    labels = kmeans.fit_predict(X)

    # Coefficient de silhouette : plus il est élevé, meilleure est la séparation des clusters
    silhouette = silhouette_score(X, labels)

    # Indice de Calinski-Harabasz : plus il est élevé, plus les clusters sont compacts et séparés
    calinski = calinski_harabasz_score(
        X,
        labels
    )

    # Indice de Davies-Bouldin : plus il est faible, meilleur est le clustering
    davies = davies_bouldin_score(
        X,
        labels
    )

    # Affichage des métriques pour comparer les valeurs de K
    print(
        f"K={k} | "
        f"Silhouette={silhouette:.4f} | "
        f"Calinski={calinski:.2f} | "
        f"Davies={davies:.4f}"
    )

import folium

# =====================================================
# 9. K-MEANS FINAL AVEC K OPTIMAL
# =====================================================

# Après analyse de la méthode du coude et des métriques, K=5 est retenu
kmeans_final = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=10
)

# Attribution d'un cluster à chaque borne
df["cluster"] = kmeans_final.fit_predict(X)

import joblib

# Sauvegarde du modèle pour le script final
# Cela évite de réentraîner le modèle à chaque utilisation
joblib.dump(kmeans_final, "modele_kmeans.pkl")

print("Modèle sauvegardé : modele_kmeans.pkl")

# Affichage du nombre de bornes dans chaque cluster
print("\nRépartition des bornes par cluster :")
print(df["cluster"].value_counts().sort_index())

# Sauvegarde du dataset avec la colonne cluster
df.to_csv("bornes_clusters_kmeans.csv", index=False)

# =====================================================
# 10. CARTE DES CLUSTERS
# =====================================================

# Création d'une carte centrée sur la France
carte = folium.Map(
    location=[46.5, 2.5],
    zoom_start=6,
    tiles="OpenStreetMap"
)

# Association d'une couleur à chaque cluster
couleurs_clusters = {
    0: "red",
    1: "blue",
    2: "green",
    3: "orange",
    4: "purple"
}

# Échantillon pour éviter une carte trop lourde à afficher
df_sample = df.sample(min(7000, len(df)), random_state=42)

# Ajout des bornes sur la carte avec une couleur selon leur cluster
for _, row in df_sample.iterrows():
    cluster = int(row["cluster"])
    couleur = couleurs_clusters.get(cluster, "gray")

    folium.CircleMarker(
        location=[
            row["consolidated_latitude"],
            row["consolidated_longitude"]
        ],
        radius=3,
        color=couleur,
        fill=True,
        fill_color=couleur,
        fill_opacity=0.7,
        popup=f"""
        <b>Cluster :</b> {cluster}<br>
        <b>Latitude :</b> {row['consolidated_latitude']}<br>
        <b>Longitude :</b> {row['consolidated_longitude']}
        """
    ).add_to(carte)

# Titre affiché sur la carte HTML
titre_html = """
<div style="
position: fixed;
top: 10px;
left: 25%;
width: 50%;
background-color: white;
border: 2px solid grey;
z-index: 9999;
text-align: center;
font-size: 22px;
font-weight: bold;
padding: 8px;">
Carte des clusters K-Means des bornes IRVE
</div>
"""

# Ajout du titre à la carte
carte.get_root().html.add_child(folium.Element(titre_html))

# Légende expliquant la couleur de chaque cluster
legende_html = """
<div style="
position: fixed;
bottom: 40px;
left: 40px;
width: 300px;
background-color: white;
border: 2px solid grey;
z-index: 9999;
font-size: 14px;
padding: 10px;">
<b>Légende - Clusters K-Means</b><br><br>
<span style="color:red;">●</span> Cluster 0<br>
<span style="color:blue;">●</span> Cluster 1<br>
<span style="color:green;">●</span> Cluster 2<br>
<span style="color:orange;">●</span> Cluster 3<br>
<span style="color:purple;">●</span> Cluster 4<br><br>
<i>Chaque couleur représente un regroupement géographique de bornes.</i>
</div>
"""

# Ajout de la légende à la carte
carte.get_root().html.add_child(folium.Element(legende_html))

# Sauvegarde de la carte interactive
carte.save("carte_clusters_kmeans.html")

print("\nCarte créée : carte_clusters_kmeans.html")
print("CSV créé : bornes_clusters_kmeans.csv")


################################################################################

import joblib
import pandas as pd

# =====================================================
# SCRIPT DE PRÉDICTION DU CLUSTER D'UNE BORNE
# =====================================================

# Chargement du modèle K-Means déjà entraîné
# Important : on ne relance pas l'entraînement ici
modele = joblib.load("modele_kmeans.pkl")

# =====================================================
# ENTRÉE UTILISATEUR
# =====================================================

# L'utilisateur saisit les coordonnées de la borne
latitude = float(input("Entrez la latitude de la borne : "))
longitude = float(input("Entrez la longitude de la borne : "))

# Création d'un tableau avec les mêmes colonnes que pendant l'entraînement
nouvelle_borne = pd.DataFrame(
    [[latitude, longitude]],
    columns=[
        "consolidated_latitude",
        "consolidated_longitude"
    ]
)

# =====================================================
# PRÉDICTION
# =====================================================

# Prédiction du cluster associé aux coordonnées saisies
cluster = modele.predict(nouvelle_borne)[0]

print("\nRésultat :")
print(f"La borne appartient au cluster {cluster}.")


###################################################################################################

import pandas as pd

# =====================================================
# CRÉATION DU FICHIER EXCEL DES VILLES
# =====================================================

# Chargement du dataset
df = pd.read_csv("export_IA.csv")

# Sélection des colonnes utiles pour associer une ville à ses coordonnées
villes = df[
    [
        "consolidated_commune",
        "consolidated_latitude",
        "consolidated_longitude"
    ]
].dropna()

# Renommage des colonnes pour rendre le fichier Excel plus lisible
villes = villes.rename(
    columns={
        "consolidated_commune": "Ville",
        "consolidated_latitude": "Latitude",
        "consolidated_longitude": "Longitude"
    }
)

# Une ville peut apparaître plusieurs fois, donc on calcule ses coordonnées moyennes
villes = (
    villes.groupby("Ville", as_index=False)
    .agg(
        Latitude=("Latitude", "mean"),
        Longitude=("Longitude", "mean")
    )
)

# Tri alphabétique pour faciliter la recherche par l'utilisateur
villes = villes.sort_values("Ville")

# Export du tableau au format Excel
villes.to_excel(
    "villes_coordonnees.xlsx",
    index=False
)

print("Fichier créé : villes_coordonnees.xlsx")